# The paper's results, replayed

Every figure and table in the paper, rebuilt from the recorded results
rather than recomputed. Nothing here trains anything; the whole
notebook runs in seconds against the published data.

If a cell below refuses, it is because the data it needs has not been
fetched. The refusal names the command that fixes it.


In [ ]:
from IPython.display import Markdown, display

from mbl.replay import (
    GateFailedError,
    StoreIncompleteError,
    load_study,
    render_figure,
    render_gates,
    render_problem,
    render_protocol,
    render_provenance,
    resolve,
)

TIER = "publication_b16k"
LARGE_TIER = "publication"

depth = load_study("icassp/fig1_depth", tier=TIER)
display(Markdown(render_problem(depth)))


## What is being compared, and how

Every controller is scored on the same plant, from the same initial
states, under the same noise, at the same numerical precision. The
comparison is fair by construction rather than by convention: the
quantities below are inference costs on held-out trajectories, never
training losses.


In [ ]:
display(Markdown(render_protocol(depth)))


## Figure 1 — cost against unfolding depth

The headline comparison. Depth is the number of unrolled iterations;
the flat series are the controllers that do not unroll and are drawn
at every depth for reference.


In [ ]:
try:
    figure_one = resolve(depth)
except (StoreIncompleteError, GateFailedError) as refusal:
    display(
        Markdown(
            "> **This figure cannot be replayed.**\n>\n> ```text\n> "
            + str(refusal).replace("\n", "\n> ")
            + "\n> ```"
        )
    )
    raise

display(Markdown(render_gates(figure_one)))


In [ ]:
display(
    Markdown(
        render_figure(
            figure_one,
            "fig1_cost_vs_depth",
            claim=(
                "Attained cost against unfolding depth, against the "
                "non-unrolling references."
            ),
        )
    )
)


The table behind the figure. Every row is one contender at one depth:
the aggregate over seeds, the spread within a seed and across seeds,
and how many trajectories each number rests on.


In [ ]:
figure_one.analyses["cost_by_depth"].table


## Figure 2 — cost against mismatch severity

Three conditions, swept over the same severity axis: the controller is
told nothing about the mismatch, told about it, or fitted in the world
that produces it. The paper draws the three as one panelled figure;
the tables below are what that figure is drawn from.


In [ ]:
conditions = {}
for condition in ("blind", "told", "world"):
    study = load_study(f"icassp/fig2_angle_{condition}", tier=TIER)
    try:
        conditions[condition] = resolve(study)
    except (StoreIncompleteError, GateFailedError) as refusal:
        display(
            Markdown(
                "> **This figure cannot be replayed.**\n>\n> ```text\n> "
                + str(refusal).replace("\n", "\n> ")
                + "\n> ```"
            )
        )
        raise

conditions["blind"].analyses["cost_by_angle_blind"].table


In [ ]:
conditions["told"].analyses["cost_by_angle_told"].table


In [ ]:
conditions["world"].analyses["cost_by_angle_world"].table


## Figure 3 — the large instance

The same depth comparison at a problem an order of magnitude larger,
where the classical alternatives become expensive. Its results were
recorded at a different effort level from the figures above, which is
why it is loaded separately.


In [ ]:
large = load_study("icassp/fig3_large_depth", tier=LARGE_TIER)
try:
    figure_three = resolve(large)
except (StoreIncompleteError, GateFailedError) as refusal:
    display(
        Markdown(
            "> **This figure cannot be replayed.**\n>\n> ```text\n> "
            + str(refusal).replace("\n", "\n> ")
            + "\n> ```"
        )
    )
    raise

display(
    Markdown(
        render_figure(
            figure_three,
            "fig3_large_cost_vs_depth",
            claim="Attained cost against unfolding depth at the large instance.",
        )
    )
)


In [ ]:
figure_three.analyses["cost_by_depth"].table


## Where these numbers came from

The block below records what produced every result above: the exact
specifications, the effort they were run at, the hardware, and the
identifiers under which each record is filed. Two runs of the same
specification resolve to the same identifier, which is what makes a
result checkable rather than merely repeatable.


In [ ]:
display(Markdown(render_provenance(figure_one)))


In [ ]:
display(Markdown(render_provenance(figure_three)))
